# Multi Agent Reinforcement Learning

While standard (single-agent) RL focuses on a _single_ agent learning to make decisions within an
environment maximizing its own cumulative reward, Multi-Agent RL (MARL) involves multiple agents
interacting within a shared environment. Each agent learns its policy, potentially influencing and
being influenced by others.

Agents might have _cooperative_ (shared reward), _competitive_ (conflicting rewards), or _mixed_
objectives, adding complexity beyond maximizing a single agent's reward.

## Key Challenges

MARL presents unique and key challenges:

- **Non-Stationarity:** The core challenge. As one agent learns and changes its policy, the
  environment effectively changes from the perspective of other agents. The learning process of
  other agents makes the environment appear dynamic and unstable to any individual agent.
- **Scalability:** The complexity (state and action spaces) can grow exponentially with the number
  of agents, making learning computationally expensive or intractable.
- **Credit Assignment:** In cooperative settings, it's difficult to determine which agent's specific
  actions contributed to the overall team success or failure, making reward distribution
  challenging.
- **Coordination/Collaboration:** Agents need to learn how to coordinate their actions effectively,
  especially with limited or no direct communication.
- **Partial Observability:** Agents often have only a limited view of the full environment state and
  may not know the internal states or intentions of other agents.

## Interesting Aspects

MARL is a very interesting setup, and deeply intertwined with **game theory**. Concepts like _Nash
Equilibrium_ (where no agent can improve its outcome by unilaterally changing its strategy) are
crucial for analyzing the stability and performance of MARL systems, especially in competitive or
mixed settings. However, finding or converging to meaningful equilibria can be difficult.

MARL systems can exhibit **complex and unexpected collective behaviors** that _emerge_ from the
interactions of individually simple agents. Studying these emergent phenomena (like flocking,
spontaneous cooperation/competition, or even simulated economic behaviors like bartering) is
fascinating and crucial for understanding system dynamics. It can also be a challenge to ensure
emergent behavior is desirable.


In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from pettingzoo.utils.env import AgentID, AECEnv
from pettingzoo.utils import aec_to_parallel
from pettingzoo.butterfly import pistonball_v6
from mpe2 import simple_adversary_v3

import supersuit as ss

from collections import deque
from dataclasses import dataclass
from typing import Type

from util.gymnastics import DEVICE, pettingzoo_simulation, init_random
from util.rl_algos import soft_update_model_params, compute_advantages_and_returns

## MPE Simple Adversary Environment

Let's use the [Simple Adversary environment](https://mpe2.farama.org/environments/simple_adversary/)
to train our MARL agents. This environment is a mixed competitive-cooperative environment, where the
two good agents need to deceive the adversary.

"_Good agents are rewarded based on how close the closest one of them is to the target landmark, but
negatively rewarded based on how close the adversary is to the target landmark. The adversary is
rewarded based on distance to the target, but it doesn’t know which landmark is the target landmark.
[...] This means good agents have to learn to ‘split up’ and cover all landmarks to deceive the
adversary._"


In [ ]:
# Environment configuration tuned for conveniency of this notebook.
MPE_SIMPLE_ADVERSARY_KWARGS = {
    "continuous_actions": True,
    "dynamic_rescaling": True,
}


def simple_adversary_v3_wrappers(zoo_env):
    """Configure wrappers on the simple adversary environment. In particular, make sure the
    observations of the good agents and the adversary have the same dimension."""
    zoo_env = ss.pad_observations_v0(zoo_env)
    return zoo_env


def create_simple_adversary_env():
    """Convenient function to setup the simple adversary environment."""
    env = simple_adversary_v3.env(**MPE_SIMPLE_ADVERSARY_KWARGS)
    env = simple_adversary_v3_wrappers(env)
    return env

In [ ]:
test_env: AECEnv = create_simple_adversary_env()
test_env.reset(seed=42)
agents: list[AgentID] = test_env.agents
print(agents)
print(test_env.action_space(agents[0]))
print(test_env.observation_space(agents[0]))

In [ ]:
pettingzoo_simulation(
    simple_adversary_v3, wrappers=[simple_adversary_v3_wrappers], **MPE_SIMPLE_ADVERSARY_KWARGS
)

## MA-DDPG

MA-DDPG is a popular algorithm for multi-agent reinforcement learning, particularly effective in
scenarios with continuous action spaces. It extends the single-agent DDPG algorithm using the
**Centralized Training with Decentralized Execution (CTDE)** paradigm.

<div>
  <img src="../assets/09_MARL_CTDE.png">
  <br>
  <small>From "Multi-Agent Actor-Critic for Mixed Cooperative-Competitive Environments"</small>
</div>

During training, a centralized controller might access all agents' observations, actions, and reward
functions to ease learning (addressing non-stationarity and credit assignment). However, during
execution, each agent acts based only on its own local observations, making the policies deployable
in real-world scenarios where centralization isn't feasible.

MA-DDPG trains a centralized critic that considers all agents' observations and actions to guide the
learning of individual actors, each of which determines a single agent's policy based only on its
local observation.

Take a look at the [paper](https://arxiv.org/abs/1706.02275)!


### Replay Buffer

Like DDPG, we use a replay buffer to store experiences. This time configured for multiple agents.


In [ ]:
class ReplayBuffer:
    """Replay buffer used in the MA-DDPG implementation.

    Stores tuples like: (obs, actions, rewards, next_obs, dones)
    """

    def __init__(self, size: int, sample_size: int, num_agents: int):
        self.size = size
        self.deque = deque(maxlen=self.size)
        self.sample_size = sample_size
        self.num_agents = num_agents

    def push(self, transition: tuple[torch.Tensor]):
        """Pushes into the buffer."""
        # TODO: Perform some sanity checks on dimensionality. The transition tuple looks like:
        #       (obs, actions, rewards, next_obs, dones)
        #       The first dimension of each tensor must always represent the agent.
        # TODO: Append the transition to the buffer.
        pass

    def sample(self) -> tuple[torch.Tensor]:
        """Samples from the buffer. Returns a tuple of tensors for convenience.

        The returned tensors shapes are: (num_agents, batch_size, ...)
        """
        # TODO: Sample from the internal `deque` a `sample_size` subset of transitions.
        samples = None

        # TODO: From the list of samples, extract each component. Use torch.stack.
        obs = None
        actions = None
        rewards = None
        next_obs = None
        dones = None

        # Create a tuple of samples. Note: dimension is (batch_size, num_agents, ...)
        t_samples = (obs, actions, rewards, next_obs, dones)
        # Transpose so each agent's network gets its own (batch_size, ...) slice.
        # TODO: Return a tuple of samples of dimension: (num_agents, batch_size, ...)
        return None

    def __len__(self):
        # TODO: Return the length of the internal deque.
        pass

### Simple Adversary Neural Networks

Configure actor and critic networks, as well as an `AgentNetwork` grouping together actor, critic,
and corresponding target networks.


In [ ]:
HIDDEN_DIM = 256


class MpeActorNetwork(nn.Module):
    """The agent policy network (actor)."""

    def __init__(self, state_dim, action_dim):
        super().__init__()
        # TODO: Configure a standard neural network with three linear layers (from state to action).

    def forward(self, x):
        # TODO: relu, relu and... the action space is [0, 1]: which final non-linearity?
        return None


class MpeCriticNetwork(nn.Module):
    """The agent centralized critic network (getting all observations)."""

    def __init__(self, state_dim, action_dim, num_agents):
        super().__init__()
        # TODO: Compute the centralized critic input size. Remember: the critic takes all agents'
        #       observations and actions into account!
        self.critic_input_size = None
        # TODO: Configure another 3 layers neural network, that outputs the Q value.
        pass

    def forward(self, obs_all_agents, actions_all_agents):
        # TODO: First, compute the centralized critic inputs. The dimension of the observation (and
        #       action) tensors are: (N_agents, Batch_size, ...).
        x = None
        # TODO: Permute the tensor to have batch dimension first: (Batch_size, N_agents, ...).
        x = None
        # TODO: Here we go, relu, relu and... do not apply any non-linearity in the last layer!
        return None

In [ ]:
class AgentNetwork(nn.Module):
    """Convenient network to encapsulate both actor, critic, and target networks for an agent."""

    def __init__(
        self,
        num_agents,
        state_dim,
        action_dim,
        # The `actor_net` needs to be instantiated via: actor_net(state_dim, action_dim).
        actor_net: Type[nn.Module],
        # The `critic_net` needs to be instantiated via: critic_net(state_dim, action_dim).
        critic_net: Type[nn.Module],
        lr_actor=1e-4,
        lr_critic=1e-3,
    ):
        super().__init__()
        self.action_dim = action_dim
        self.num_agents = num_agents

        # TODO: Setup the actor network
        self.actor = None
        # TODO: Setup the actor target network
        self.actor_target = None
        # TODO: Copy the parameters of the actor network into the actor target network. Also, make
        #       sure to set the target network into eval mode.
        # ...
        # TODO: Created the Adam optimizer for the actor network (eps=1e-5)
        self.actor_optimizer = None

        # TODO: Setup the critic network
        self.critic = None
        # TODO: Setup the critic target network
        self.critic_target = None
        # TODO: Copy the parameters of the critic network into the critic target network. Also, make
        #       sure to set the target network into eval mode.
        # ...
        # TODO: Created the Adam optimizer for the critic network (eps=1e-5)
        self.critic_optimizer = None

### MA-DDPG Agent

MA-DDPG implementation which resembles the one described in the original paper. The actor / critic
networks are generic, so that we can plug in different architecture for different environments (see
`Pistonball` below).


In [ ]:
class MA_DDPG:
    """The MA-DDPG algorithm: multi-agent training!"""

    def __init__(
        self,
        actor_net: Type[nn.Module],
        critic_net: Type[nn.Module],
        agent_ids,
        state_dim,
        action_dim,
        action_min=-1,
        action_max=1,
        gamma=0.99,
        min_buffer_size=2048,
        buffer_len=int(1e5),
        batch_size=128,
        update_every=8,
        n_updates=4,
        update_policy_every=2,
        noise_start=1.0,
        noise_reduction=0.99995,
        noise_min=0.25,
        max_grad_norm=0.5,
        lr_actor=1e-4,
        lr_critic=1e-3,
        tau=0.01,
    ):
        self.num_agents = len(agent_ids)
        self.agent_id_map = {id: index for (index, id) in enumerate(agent_ids)}
        self.agent_index_map = {index: id for (index, id) in enumerate(agent_ids)}
        self.action_dim = action_dim
        self.action_min = action_min
        self.action_max = action_max
        self.gamma = gamma
        self.noise = noise_start
        self.noise_reduction = noise_reduction
        self.noise_min = noise_min
        self.batch_size = batch_size
        self.min_buffer_size = min_buffer_size
        self.n_updates = n_updates
        self.update_every = update_every
        self.update_policy_every = update_policy_every
        self.max_grad_norm = max_grad_norm
        self.tau = tau
        self.t_step = 0

        # TODO: Create the ReplayBuffer.
        self.replay_buffer = None

        # TODO: Create N agents (AgentNetwork) for the multi-agent algorithm.
        self.agents = None

    def step(self, obs, actions, next_obs, rewards, dones):
        """Multi-agent step: tracks the experience, and updates the agents using batch learning."""
        # Convert dicts keyed by agent ID to lists/tensors ordered by agent index. This is helpful
        # to convert observations, etc. from PettingZoo ParallelEnv to a format suited to our MADDPG
        # algorithm implementation.
        convert = lambda t: [t[self.agent_index_map[i]] for i in range(self.num_agents)]

        # TODO: Convert inputs and create corresponding tensors.
        obs_t = None
        actions_t = None
        next_obs_t = None
        rewards_t = None
        dones_t = None  # Hint: uint8

        # TODO: Store transition in the replay buffer.
        # ...
        # TODO: Increment global timestamp!
        self.t_step = None

        # Learning Step
        if len(self.replay_buffer) > self.min_buffer_size and self.t_step % self.update_every == 0:
            # TODO: Update noise with exponential decay.
            self.noise = None
            for cycle in range(self.n_updates):
                print(f"Learning cycle {cycle+1}/{self.n_updates}...".ljust(100), end="\r")
                # TODO: Sample a batch of transitions from the replay buffer.
                samples = None
                obs, actions, rewards, next_obs, dones = None
                for a_i in range(len(self.agents)):
                    # TODO: Perform critic update. Hint: use method defined below.
                    # TODO: If is_actor_update(), perform actor update.
                    pass
            # TODO: Soft update the target networks towards the actual networks. IMPORTANT: Update
            #       the target networks _after_ all the updates (which use target predictions
            #       themselves). Hint: use the `update_targets` method below.
            pass

    def learn_actor(self, agent_number: int, obs):
        """Perform a single gradient descent step on the actor of the selected agent.

        The `obs` parameter contains the observations from _all_ agents.
        """
        # TODO: Get the agent.
        agent = None
        # TODO: Get _all_ action preditionsfrom all the agents. Make sure to detach() all the other
        #       agent predictions!
        actions_pred = None
        # TODO: Compute the actor_loss as the negative value computed by the critic.
        actor_loss = None

        # TODO: Perform an optimizer step. Zero-grad, backward, step. Make sure to clip gradients
        #       Before calling step(). Hint: Use clip_grad_norm_.
        pass

    def learn_critic(self, agent_number: int, obs, actions, rewards, next_obs, dones):
        """Perform a single gradient descent step on the critic of the selected agent.

        The input tensors contains the observations, actions, etc. from _all_ agents.
        """
        # TODO: Get the agent.
        agent = None
        with torch.no_grad():
            # TODO: Get predicted all next-state actions and Q values from target models.
            actions_next = None
            # TODO: Get the next Q targets from target model for this agent.
            Q_targets_next = None
            # TODO: Compute Q targets for current states. Remember:
            #       Q_target = reward + gamma * next_target * (1 - done)
            Q_targets = None

        # TODO: Compute the Q_expected using the critic.
        Q_expected = None
        # TODO: Calculate the critic loss (MSE).
        critic_loss = None

        # TODO: Perform an optimizer step. Zero-grad, backward, step. Make sure to clip gradients
        #       Before calling step(). Hint: Use clip_grad_norm_.
        pass

    def update_targets(self):
        """Soft-update target networks."""
        for agent in self.agents:
            # TODO: Update critic target parameters. If is_actor_update(), update actor target
            #       parameters. Hint: use `soft_update_model_params`.
            pass

    @torch.no_grad()
    def eval_act(self, agent_id: AgentID, obs: np.ndarray, add_noise=False):
        """Get action from the actor network for evaluation/execution."""
        # TODO: Convert the obs into a tensor. Make sure to add the batch dimension (unsqueeze).
        obs_t = None
        # TODO: Get the agent_number from the agent_id_map
        agent_number = None
        # TODO: Lookup the agent.
        agent = None

        # TODO: Compute the action. Hint: actor network to the rescue.
        action = None

        if add_noise:
            # TODO: Add noise. Hint: use the get_noise() method below.
            action += None

        # TODO: Convert the action to numpy. Make sure to remove the batch dimension.
        action = None

        # Return the action, clipped between [action_min, action_max] for safety.
        return None

    def is_actor_update(self) -> bool:
        """Determines the actor network should be updated in the current global timestep."""
        return self.t_step % (self.update_every * self.update_policy_every) == 0

    def get_noise(self) -> torch.Tensor:
        """Compute the Gaussian noise."""
        # TODO: Return the gaussian noise. Hint: use np.random.normal.
        return None

### Training Loop

MA-DDPG training loop. It expects a PettingZoo [AECEnv](https://pettingzoo.farama.org/api/aec/) but
converts it into [parallel](https://pettingzoo.farama.org/api/parallel/) right away for convenience
of training this notebook MA-DDPG algorithm implementation.


In [ ]:
def train_maddpg(aec_env: AECEnv, agent: MA_DDPG = None, max_episodes=1_000):
    """Main training loop for MA-DDPG."""
    # TODO: Convert the AECEnv to ParallelEnv for ease of training MA-DDPG with this notebook
    #       specific setup. Hint: use the `aec_to_parallel` utility.
    env = None

    returns = []
    timestep = 0
    env_agent_ids = aec_env.agents
    # Initialize so the trailing print never raises UnboundLocalError when `max_episodes < 25`
    # and the periodic logging block below never runs.
    avg_return = float("nan")

    print(f"Training started. Collecting initial experiences...".ljust(100), end="\r")
    for n_episode in range(1, max_episodes + 1):
        # TODO: Initialize a dictionary of {agent_id: 0.0} to record episode returns.
        episode_returns = None
        # TODO: Reset the environment.
        obs, _ = None

        # PettingZoo removes agents from env.agents when those agents have completed their episode.
        while env.agents:
            # TODO: Get all the actions from all agents as a dictionary of {agent_id: action}
            actions = None

            # TODO: Perform a single step in the parallel environment.
            next_obs, rewards, terms, truncs, _ = None

            # TODO: Compute whether agents have completed as a dictionary of agent_id: done.
            #       Hint: an agent is done if terminated or truncated.
            dones = None

            # TODO: Call step(...) on the agent!
            pass

            # TODO: Move to the next observation... easy to overlook!
            obs = None

            # TODO: Update timestep, and all agent episode returns
            timestep += None
            for agent_id in env_agent_ids:
                episode_returns[agent_id] += None

        # Record the current episode returns.
        returns.append(episode_returns)

        # Convenient log message every N episodes.
        if n_episode % 25 == 0:
            last_100_returns = returns[-100:]
            avg_returns_by_agent = {id: 0.0 for id in env_agent_ids}
            for ret in last_100_returns:
                for id in env_agent_ids:
                    avg_returns_by_agent[id] += ret[id]
            avg_returns_by_agent = {
                key: value / len(last_100_returns) for key, value in avg_returns_by_agent.items()
            }
            avg_return = np.mean([v for v in avg_returns_by_agent.values()])
            formatted_returns = (
                "{"
                + ", ".join(f"{repr(k)}: {v:.3f}" for k, v in avg_returns_by_agent.items())
                + "}"
            )
            print(f"[Episode {n_episode} AVG: {avg_return:.3f}] {formatted_returns}".ljust(100))

    print(f"Episode {n_episode} terminating training, average reward: {avg_return:.3f}".ljust(100))
    return returns

## Train Simple Adversary

We can train the `simple_adversary_v3` environment with MA-DDPG and obtain policies that show good
agents trying to trick the adversary and the adversary trying to become smarter. MA-DDPG has various
limitations, some already notable with this MPE environment:

- **Sensitivity to hyperparameters:** Finding the right learning rates, exploration parameters, and
  network architectures often requires extensive tuning.
- **Emergent Unwanted Behaviors:** Agents might learn suboptimal or even adversarial strategies that
  exploit the weaknesses of other agents or the environment, even if these behaviors weren't
  explicitly intended, landing in suboptimal policies. Sometimes early stopping is used.
- **Scalability Issues:** As the number of agents increases, the size of the joint
  action-observation space grows exponentially, making it computationally expensive and sample
  inefficient to learn joint policies. We can see this in the _Pistonball_ environment below.


In [ ]:
def plot_agent_returns(episode_returns):
    """Convenient function to plot agents and rolling average returns."""
    df = pd.DataFrame(episode_returns)
    agent_ids = df.columns.tolist()
    window_size = 100

    for agent_id in agent_ids:
        df[f"{agent_id}_running_avg"] = (
            df[agent_id].rolling(window=window_size, min_periods=1).mean()
        )
    df["overall_average_return"] = df[agent_ids].mean(axis=1)
    df["running_overall_average_return"] = (
        df["overall_average_return"].rolling(window=window_size, min_periods=1).mean()
    )

    plt.figure(figsize=(10, 3.7))
    colors = cm.get_cmap("tab10", len(agent_ids))
    for i, agent_id in enumerate(agent_ids):
        agent_color = colors(i)
        plt.plot(
            df.index,
            df[agent_id],
            label=f"{agent_id} Raw Return",
            alpha=0.3,
            color=agent_color,
            linestyle=":",
        )
        plt.plot(
            df.index,
            df[f"{agent_id}_running_avg"],
            label=f"{agent_id} Running Avg",
            color=agent_color,
            linewidth=1.5,
        )
    plt.plot(
        df.index,
        df["running_overall_average_return"],
        label="Running Avg (All Agents)",
        color="black",
        linewidth=2.5,
        linestyle="--",
    )

    plt.xlabel("Episode")
    plt.ylabel("Return")
    plt.title("Agent Returns: Raw, Individual Running Averages, and Overall Running Average")
    plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
    plt.grid(True)
    plt.tight_layout(rect=[0, 0, 0.85, 1])
    plt.show()

### Basic learning

With `~7k` episodes and a first set of hyperparameters, we see the agents trying to distance the
adversary when it gets too close to the target.

NOTE: I didn't spend too much time in configuring the hyperparameters and/or tailor the MA-DDPG
implementation to this MPE environment (e.g., using the
[Ornstein Uhlenbeck noise](https://en.wikipedia.org/wiki/Ornstein%E2%80%93Uhlenbeck_process) as in
the original paper). Hence, it is very possible that we can train this environment much faster.


In [ ]:
ad_env: AECEnv = create_simple_adversary_env()
ad_env = init_random(ad_env)
agent_ids = ad_env.possible_agents
state_dim = ad_env.observation_space(agent_ids[0]).shape[0]
action_dim = ad_env.action_space(agent_ids[0]).shape[0]

agent_adv_basic = MA_DDPG(
    MpeActorNetwork,
    MpeCriticNetwork,
    agent_ids,
    state_dim,
    action_dim,
    action_min=0,
    gamma=0.995,
    min_buffer_size=2048,
    buffer_len=int(3e5),
    batch_size=128,
    update_every=50,
    n_updates=8,
    update_policy_every=4,
    noise_start=0.5,
    noise_reduction=0.99995,
    noise_min=0.1,
    lr_actor=5e-5,
    lr_critic=5e-4,
    max_grad_norm=0.1,
    tau=1e-3,
)

In [ ]:
# 25m training on GPU. Consider running on CPU given that CPU/GPU communication is bottleneck.
adv_scores_basic = train_maddpg(ad_env, agent_adv_basic, max_episodes=7_500)

In [ ]:
plot_agent_returns(adv_scores_basic)

In [ ]:
pettingzoo_simulation(
    simple_adversary_v3,
    agent=agent_adv_basic,
    seed=100,
    wrappers=[simple_adversary_v3_wrappers],
    **MPE_SIMPLE_ADVERSARY_KWARGS
)

### Better strategy

With `30k+` episodes and a different set of hyperparameters, we see how they tend to learn how to
split apart as a better strategy.


In [ ]:
ad_env: AECEnv = create_simple_adversary_env()
ad_env = init_random(ad_env)
agent_ids = ad_env.possible_agents
state_dim = ad_env.observation_space(agent_ids[0]).shape[0]
action_dim = ad_env.action_space(agent_ids[0]).shape[0]

agent_adv_better = MA_DDPG(
    MpeActorNetwork,
    MpeCriticNetwork,
    agent_ids,
    state_dim,
    action_dim,
    action_min=0,
    gamma=0.95,
    min_buffer_size=2048,
    buffer_len=int(3e5),
    batch_size=128,
    update_every=16,
    n_updates=4,
    update_policy_every=2,
    noise_start=1.0,
    noise_reduction=0.99995,
    noise_min=0.25,
)

In [ ]:
# 2h training on GPU. Consider running on CPU given that CPU/GPU communication is bottleneck.
adv_scores_better = train_maddpg(ad_env, agent_adv_better, max_episodes=30_000)

In [ ]:
plot_agent_returns(adv_scores_better)

In [ ]:
pettingzoo_simulation(
    simple_adversary_v3,
    agent=agent_adv_better,
    seed=100,
    wrappers=[simple_adversary_v3_wrappers],
    **MPE_SIMPLE_ADVERSARY_KWARGS
)

## Pistonball v6

Pistonball is a "_physics based cooperative game where the goal is to move the ball to the left-wall
of the game border by activating the vertically moving pistons. To achieve an optimal policy for the
environment, pistons must learn highly coordinated behavior_".

Pistonball presents difficulties for MADDPG because coordinating and scaling a large number of
agents with high-dimensional local image observations is complex, and the shared reward structure
makes it hard for the centralized critic to accurately assign credit for collective success or
failure to individual agents, hindering the learning of precise, coordinated actions.


In [ ]:
PISTONBALL_N5 = {
    "n_pistons": 5,
    "max_cycles": 20,
    "time_penalty": -0.3,
}

FRAME_W = 32
FRAME_H = 32


def pistonball_v6_wrappers(zoo_env):
    zoo_env = ss.color_reduction_v0(zoo_env, mode="B")
    zoo_env = ss.resize_v1(zoo_env, FRAME_W, FRAME_H)
    zoo_env = ss.frame_stack_v1(zoo_env, stack_size=3)
    zoo_env = ss.dtype_v0(zoo_env, "uint8")
    return zoo_env


def create_pistonball_env(**env_kwargs):
    zoo_env: AECEnv = pistonball_v6.env(**env_kwargs)
    zoo_env = pistonball_v6_wrappers(zoo_env)
    return zoo_env

In [ ]:
test_env: AECEnv = create_pistonball_env()
test_env.reset(seed=42)
agents: list[AgentID] = test_env.agents
print(agents)
print(test_env.action_space(agents[0]))
print(test_env.observation_space(agents[0]))

In [ ]:
pettingzoo_simulation(pistonball_v6, wrappers=[pistonball_v6_wrappers], **PISTONBALL_N5)

### Pistonball Neural Networks

Unlike the MPE environment (where observations are small state vectors), Pistonball gives each
piston a local **image observation** — a cropped view of the game area around it. This means we need
a convolutional feature extractor to process the raw pixels into a compact feature vector before
feeding it into the actor and critic heads. We reuse the same `convolutional_feature_extractor` for
both MADDPG and MAPPO below.


In [ ]:
FEAT_SIZE = 256
N_STACKED_FRAMES = 3


def convolutional_feature_extractor(output_dim) -> nn.Module:
    return nn.Sequential(
        nn.Conv2d(N_STACKED_FRAMES, 32, 3, padding=1),
        nn.MaxPool2d(2),
        nn.ReLU(),
        nn.Conv2d(32, 64, 3, padding=1),
        nn.MaxPool2d(2),
        nn.ReLU(),
        nn.Conv2d(64, 128, 3, padding=1),
        nn.MaxPool2d(2),
        nn.ReLU(),
        nn.Flatten(),
        nn.Linear(128 * 4 * 4, output_dim),
        nn.ReLU(),
    )


class PistonballActorNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.features = convolutional_feature_extractor(FEAT_SIZE)
        self.fc1 = nn.Linear(FEAT_SIZE, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, action_dim)

    def forward(self, x):
        x = x.float() / 255.0
        x = x.permute(0, 3, 1, 2)
        x = self.features(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return F.tanh(self.fc3(x))


class PistonballCriticNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, num_agents):
        super().__init__()
        self.num_agents = num_agents
        self.critic_input_size = (FEAT_SIZE + action_dim) * num_agents
        self.features = convolutional_feature_extractor(FEAT_SIZE)
        self.fc1 = nn.Linear(self.critic_input_size, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 1)

    def forward(self, obs_all_agents, actions_all_agents):
        # action_all_agents shape (n_agents, batch_size, action_dim)
        actions_all_agents = actions_all_agents.permute(1, 0, 2)
        x = obs_all_agents.float() / 255.0
        x = x.permute(1, 0, 4, 2, 3)
        batch_size, num_agents, channels, height, width = x.shape

        x = x.reshape(batch_size * num_agents, channels, height, width)
        x = self.features(x)  # (batch_size * num_agents, feature_size)
        x = x.reshape(batch_size, num_agents, -1)  # (batch_size, num_agents, feature_size)

        x = torch.cat((x, actions_all_agents), dim=2).float().to(DEVICE)
        x = x.flatten(1, 2)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

### Training 5 Pistons

In training just 5 pistons, MADDPG converged to a suboptimal policy where the second-to-right-most
piston consistently moves up while the rest remain down. This simple coordinated behavior likely
provides just enough upward force or a consistent bounce point to guide the ball towards the target
zone, resulting in some positive reward. However, this strategy is far from optimal because true
mastery of Pistonball requires dynamic, precise, and coordinated movements from all pistons to
effectively control the ball. This convergence to a local optimum can happen in MADDPG due to the
challenges of credit assignment in shared reward settings; the simple "second piston up" strategy
consistently yields some reward, reinforcing this basic coordination and potentially preventing the
algorithm from exploring and discovering the more complex, highly coordinated behaviors needed for
optimal performance with multiple interacting agents.


In [ ]:
pv6_env: AECEnv = create_pistonball_env(**PISTONBALL_N5)
pv6_env = init_random(pv6_env)
agent_ids = pv6_env.possible_agents
state_dim = pv6_env.observation_space(agent_ids[0]).shape[0]
action_dim = pv6_env.action_space(agent_ids[0]).shape[0]

agent_n5 = MA_DDPG(
    PistonballActorNetwork,
    PistonballCriticNetwork,
    agent_ids,
    state_dim,
    action_dim,
    gamma=0.999,
    min_buffer_size=10_000,
    buffer_len=int(110_000),
    batch_size=128,
    noise_start=1.0,
    noise_reduction=(1.0 - 5e-6),
    noise_min=0.5,
    update_every=32,
    n_updates=8,
    update_policy_every=2,
    lr_actor=1e-5,
    lr_critic=5e-5,
)

In [ ]:
# 8 minutes on GPU.
p5_scores = train_maddpg(pv6_env, agent_n5, max_episodes=1500)

In [ ]:
plot_agent_returns(p5_scores)

In [ ]:
pettingzoo_simulation(
    pistonball_v6, agent=agent_n5, wrappers=[pistonball_v6_wrappers], **PISTONBALL_N5
)

## MA-PPO

In the MADDPG section above we used _CTDE_ (Centralized Training, Decentralized Execution) by giving
each critic access to all agents' observations and actions. That works in small environments, but in
Pistonball MADDPG often becomes harder to train as the number of agents grows.

A practical alternative is to switch from an off-policy actor-critic (DDPG-style) to an on-policy
method (PPO-style):

- **PPO updates are conservative** thanks to the clipping objective, which helps stability.
- In cooperative environments with **homogeneous agents** (same role, same observation/action
  shapes), it is often enough to use **a shared policy** (parameter sharing) across all agents. This
  dramatically reduces the parameter count and lets every agent's experience improve the same
  network — a form of implicit data augmentation.
- The on-policy + shared-policy combination is commonly called **MAPPO** (Multi-Agent PPO).

It differs from **IPPO (Independent PPO)** where each agent runs PPO with its _own_ network
parameters. While IPPO is a strong baseline, parameter-shared MAPPO leverages homogeneity for better
sample efficiency and avoids the moving-target problem of independently-updating actors.
[MAPPO](https://arxiv.org/abs/2103.01955) is a well-regarded and often high-performing choice for
cooperative tasks requiring coordinated control.

**A note on parameter sharing.** In our MAPPO implementation every piston runs the _same_ actor
network (shared weights). This is a deliberate choice: Pistonball pistons are _homogeneous_ — they
have the same observation space, action space, and role. Sharing parameters dramatically reduces the
number of learnable weights (one policy instead of $N$) and lets every agent's experience improve
the shared policy. When agents are _heterogeneous_ (different roles, observation spaces, or action
spaces), parameter sharing can hurt because a single network cannot specialize. In those cases you
would use per-agent policies (as in our MADDPG above) or methods like HAPPO that handle
heterogeneity explicitly.

**A note on the critic.** The original MAPPO paper (Yu et al. 2022) uses a _centralized critic_
$V_\phi(s_{\text{global}})$ that takes the full state of the environment. For educational clarity we
simplify this to a **per-agent local critic** $V_\phi(o_i)$ — each agent estimates its value from
its own local observation, with parameters shared across agents. This is closer in spirit to IPPO
than to paper MAPPO. The simplification keeps actor and critic network shapes identical and avoids
the engineering of constructing a "global state" representation. We discuss the trade-off briefly in
the results section.


### MAPPO (with parameter sharing) in one picture

We keep **one actor** (shared weights) that each piston runs locally:

$$
a_i \sim \pi_\theta(a_i \mid o_i)
$$

We use a **per-agent local critic** with parameters shared across agents:

$$
V_\phi(o_i) \approx \mathbb{E}\left[\sum_{t \ge 0} \gamma^t r_{t} \mid o_0 = o_i \right]
$$

This is a simplification compared to paper MAPPO's centralized critic, but it keeps the network
shapes identical between actor and critic and avoids the engineering complexity of constructing a
"global state" from per-agent observations. The on-policy PPO updates plus parameter sharing are
already enough to outperform MADDPG on Pistonball, as we'll see.


### Important notes (and simplifications)

- **Action space**: `pistonball_v6` defaults to _continuous_ actions in $[-1, 1]$. We use a simple
  diagonal-Gaussian policy with state-independent `log_std` (a `nn.Parameter`), and clip sampled
  actions to $[-1, 1]$ before passing them to the env. This is the standard CleanRL-style choice for
  continuous PPO — no `tanh`-squashing tricks needed.
- **Local critic**: as noted above, our critic is per-agent $V_\phi(o_i)$ with parameters shared,
  not a centralized $V_\phi(s_{\text{global}})$.
- **Entropy bonus**: the diagonal-Gaussian entropy is closed-form
  ($\mathcal{H} = \log\sigma +
  \tfrac12 \log(2\pi e)$, summed over action dims) and we use it
  directly.
- **Reward distribution**: Pistonball v6 gives every agent the _same_ global reward per step (shared
  cooperative reward — there is no `local_ratio` parameter to control this in v6). After the
  `pettingzoo_env_to_vec_env_v1` wrapper this becomes one reward per agent slot, all numerically
  identical for slots from the same env copy.


### Some utility functions first


> **Numerical-stability note: σ runaway and the `log_std` clamp.**
>
> Continuous PPO with a state-independent `log_std` parameter and an entropy bonus has a silent
> failure mode: `log_std` can drift unboundedly upward.
>
> The Gaussian entropy is $\mathcal{H} = \log\sigma + \tfrac12\log(2\pi e)$, so any positive entropy
> bonus pushes `log_std` upward — and there is no natural counterforce. As `log_std` climbs:
>
> 1. Sampled actions become wider, eventually exceeding $[-1, 1]$ on most samples and getting
>    clipped to bang-bang ±1 by the env wrapper.
> 2. The policy gradient on the mean $\mu$ scales with $1/\sigma^2$, so as $\sigma$ grows the actor
>    receives weaker and weaker gradient signal on $\mu$. The mean stops learning.
> 3. The deterministic policy ($\mu$ alone) becomes degenerate — close to whatever it happened to be
>    when σ ran away — while the stochastic policy keeps producing decent training returns from pure
>    noise.
>
> The fix in our implementation is the simplest possible one: a **hard clamp** on `log_std` applied
> directly to the parameter after each optimizer step:
>
> ```python
> self.actor.log_std.data.clamp_(max=0.0)
> ```
>
> This caps $\sigma \le 1$. Gradients flow normally through `log_std`; the cap only kicks in when
> the optimizer would push `log_std` above $0$. No smooth re-parameterization, no `tanh` bounding —
> just a one-line in-place clamp.
>
> This is a real-world MARL gotcha worth knowing about — without the clamp, `log_std` climbed past
> $+3$ (i.e., $\sigma \approx 20$) in our early Pistonball-20 experiments before we diagnosed it.


In [ ]:
# Small utility for moving inputs to the right device. The policy is a diagonal Gaussian
# (`torch.distributions.Normal`) whose sample is then **tanh-squashed** so the action it
# executes is bounded in (-1, 1). The corresponding log-prob includes the standard
# change-of-variables correction `-log(1 - tanh(u)^2)`. This keeps the importance-sampling
# ratio honest in PPO: the same random variable (the pre-squash Gaussian sample) is used
# both at collection and at update time.
def _as_torch(x, device=DEVICE, dtype=torch.float32):
    if isinstance(x, torch.Tensor):
        return x.to(device=device, dtype=dtype)
    return torch.as_tensor(x, device=device, dtype=dtype)

### Networks


In [ ]:
def _layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    """Orthogonal init with optional gain; CleanRL convention.

    Use std=sqrt(2) for hidden layers (ReLU), std=0.01 for the policy mean
    head (keeps initial actions small), std=1.0 for the value head.
    """
    nn.init.orthogonal_(layer.weight, std)
    nn.init.constant_(layer.bias, bias_const)
    return layer


class MAPPOActor(nn.Module):
    """Diagonal Gaussian policy with a CNN trunk, shared across agents."""

    def __init__(self, action_dim, feat_dim=FEAT_SIZE):
        super().__init__()
        self.features = convolutional_feature_extractor(feat_dim)
        # Re-init the last Linear inside the feature extractor (the one
        # before the final ReLU). Conv layers keep PyTorch default init.
        _layer_init(self.features[-2])

        self.fc = nn.Sequential(
            _layer_init(nn.Linear(feat_dim, 256)),
            nn.ReLU(),
            _layer_init(nn.Linear(256, 128)),
            nn.ReLU(),
        )
        # Small gain on the policy head so initial actions cluster near 0.
        self.mean = _layer_init(nn.Linear(128, action_dim), std=0.01)

        # State-independent log_std (standard PPO choice for continuous
        # actions), initialized at 0 so std=1 at the start of training.
        self.log_std = nn.Parameter(torch.zeros(action_dim))

    def forward(self, obs_batch):
        """obs_batch: (B, H, W, C) -> mean (B, action_dim), log_std (B, action_dim)"""
        # TODO: Implement the forward pass.
        # 1. Convert uint8 obs to float and scale to [0, 1].
        # 2. Permute from (B, H, W, C) to (B, C, H, W) for PyTorch CNNs.
        # 3. Run through self.features then self.fc.
        # 4. Compute the mean head output.
        # 5. Expand self.log_std to match the batch shape of mean.
        # 6. Return (mean, log_std).
        mean = None
        log_std = None
        return mean, log_std


class MAPPOCritic(nn.Module):
    """Per-agent value V(o_i) with shared parameters across agents.

    NOTE: The original MAPPO paper uses a centralized critic V(s_global). We
    use a per-agent local critic for educational simplicity — the network
    shapes match the actor's, and we avoid the engineering of constructing a
    "global state" representation.
    """

    def __init__(self, feat_dim=FEAT_SIZE):
        super().__init__()
        self.features = convolutional_feature_extractor(feat_dim)
        _layer_init(self.features[-2])

        self.v = nn.Sequential(
            _layer_init(nn.Linear(feat_dim, 256)),
            nn.ReLU(),
            _layer_init(nn.Linear(256, 128)),
            nn.ReLU(),
            # Gain of 1 on the value head (standard CleanRL choice).
            _layer_init(nn.Linear(128, 1), std=1.0),
        )

    def forward(self, obs):
        """obs: (B, H, W, C) -> values: (B,)"""
        # TODO: Implement the forward pass.
        # Same image preprocessing as the actor, then run through self.v.
        # Squeeze the trailing dim of size 1 so the output is (B,).
        return None

### Algorithm


In [ ]:
@dataclass
class MAPPOHyperParams:
    gamma: float = 0.99
    gae_lambda: float = 0.9

    clip_coef: float = 0.4

    # Entropy bonus. For continuous actions, Gaussian entropy scales with
    # log σ, so any ent_coef > 0 is a permanent upward pressure on log_std
    # with no natural counterforce — σ drifts unbounded, actions clip to
    # bang-bang noise, and μ stops receiving useful gradient signal.
    # CleanRL's continuous PPO default is 0.0 for exactly this reason.
    ent_coef: float = 0.1

    # CleanRL's standard continuous-PPO default is 3e-4 with linear annealing,
    # but we found it too aggressive on Pistonball — the noisy gradient signal
    # combined with large updates drives the policy past good local optima.
    # We use 2e-5 (matching the RLlib Pistonball tutorial) with linear annealing
    # to 0 so late-stage updates stop pushing the policy around once it's
    # reached a stable region.
    lr_actor: float = 2e-5
    lr_critic: float = 2e-5
    anneal_lr: bool = True

    ppo_epochs: int = 10
    minibatch_size: int = 2048
    max_grad_norm: float = 0.5

    # Parallel rollout settings (CleanRL-style). With n_envs=10 copies and
    # n_agents=5, each env.step() advances 50 agent slots. rollout_steps=128
    # is comfortably more than one Pistonball-N5 episode (max_cycles=20) per
    # copy per update, giving 128 * 50 = 6_400 samples per PPO update from
    # ~64 completed episodes — dense gradient signal for a small env.
    n_envs: int = 10
    rollout_steps: int = 128

In [ ]:
class MAPPO:
    """MAPPO with parameter sharing and per-agent advantages."""

    def __init__(self, agent_ids, action_dim, hp: MAPPOHyperParams = None):
        self.agent_ids = list(agent_ids)
        self.n_agents = len(self.agent_ids)
        self.action_dim = action_dim
        self.hp = hp or MAPPOHyperParams()

        # TODO: Create the (parameter-shared) actor and critic on DEVICE,
        # then create Adam optimizers for each.
        self.actor = None
        self.critic = None
        self.opt_actor = None
        self.opt_critic = None

    @staticmethod
    def _tanh_squash(pre_squash):
        """Tanh-squash a pre-squash Gaussian sample and return (action, log_det).

        `log_det` is the change-of-variables correction we subtract from the Gaussian
        log-prob so the PPO importance ratio refers to the *executed* action.
        """
        action = torch.tanh(pre_squash)
        log_det = torch.log(1.0 - action.pow(2) + 1e-6).sum(dim=-1)
        return action, log_det

    @torch.no_grad()
    def eval_act(self, agent_id, obs, add_noise=False):
        """Used by pettingzoo_simulation(...) for visualization.

        Defaults to deterministic tanh(μ); pass add_noise=True for stochastic.
        """
        self.actor.eval()
        obs_t = _as_torch(obs).unsqueeze(0)
        mean, log_std = self.actor(obs_t)
        if add_noise:
            std = torch.exp(log_std)
            pre_squash = torch.distributions.Normal(mean, std).sample()
        else:
            pre_squash = mean
        # Squash to the env's [-1, 1] action range; matches the training-time policy.
        action = torch.tanh(pre_squash)
        return action.squeeze(0).cpu().numpy()

    def _act_and_logprob(self, obs_all):
        """obs_all: (B, H, W, C) -> action (B, A), pre_squash (B, A), logp (B,).

        We squash the Gaussian sample with tanh so the executed action is bounded.
        The buffer stores `pre_squash` (the Gaussian sample) — that's what
        `update()` re-evaluates the log-prob on under the new policy.
        """
        # TODO: Build a Normal distribution from (mean, exp(log_std)) and sample
        # the pre-squash sample (`pre_squash`). Then call `self._tanh_squash` to get
        # `action` and `log_det`. The Gaussian-side log-prob is `dist.log_prob(pre_squash)
        # .sum(-1)`; subtract `log_det` to get the corrected `logp`.
        pre_squash = None
        action = None
        logp = None
        return action, pre_squash, logp

    def update(self, b_obs, b_actions, b_logprobs, b_advantages, b_returns):
        """PPO update from flattened rollout tensors. `b_actions` holds the
        *pre-squash* Gaussian samples that the rollout buffer stored.

        Returns a diagnostics dict: {approx_kl, mean_abs_adv, log_std}.
        """
        self.actor.train()
        self.critic.train()

        mean_abs_adv = b_advantages.abs().mean().item()
        # Standard PPO trick: normalize advantages before the actor update.
        b_advantages = (b_advantages - b_advantages.mean()) / (b_advantages.std() + 1e-8)

        B = b_obs.shape[0]

        # ---- Critic update (minibatched MSE against returns) ----
        for _ in range(self.hp.ppo_epochs):
            idxs = torch.randperm(B, device=DEVICE)
            for start in range(0, B, self.hp.minibatch_size):
                mb = idxs[start : start + self.hp.minibatch_size]
                # TODO: Compute the value-function loss (MSE between
                # self.critic(b_obs[mb]) and b_returns[mb]), backprop,
                # clip gradients to self.hp.max_grad_norm, and step the
                # critic optimizer.
                pass

        # ---- Actor update (PPO clipped surrogate + entropy bonus) ----
        approx_kls = []
        for _ in range(self.hp.ppo_epochs):
            idxs = torch.randperm(B, device=DEVICE)
            for start in range(0, B, self.hp.minibatch_size):
                mb = idxs[start : start + self.hp.minibatch_size]

                mean, log_std = self.actor(b_obs[mb])
                std = torch.exp(log_std)
                dist = torch.distributions.Normal(mean, std)
                # `b_actions[mb]` are *pre-squash* samples. Re-evaluate the
                # Normal log-prob on them and subtract the tanh correction so
                # the importance ratio is consistent with collection time.
                pre_squash_mb = b_actions[mb]
                _, log_det = self._tanh_squash(pre_squash_mb)
                new_logp = dist.log_prob(pre_squash_mb).sum(-1) - log_det
                # We use the *Normal* entropy as the exploration bonus — a common
                # simplification (the post-squash entropy has no closed form).
                entropy = dist.entropy().sum(-1).mean()

                # TODO: Compute the PPO clipped surrogate objective.
                # 1. logratio = new_logp - b_logprobs[mb]
                # 2. ratio = exp(logratio)
                # 3. unclipped term: ratio * advantages
                # 4. clipped term:   clamp(ratio, 1 - clip_coef, 1 + clip_coef) * advantages
                # 5. policy_loss = -mean(min(unclipped, clipped))
                logratio = None
                ratio = None
                unclipped = None
                clipped = None
                policy_loss = None

                with torch.no_grad():
                    # CleanRL's unbiased approximate KL estimator (Schulman blog).
                    approx_kls.append(((ratio - 1) - logratio).mean().item())

                # TODO: total loss = policy_loss - ent_coef * entropy.
                # Then zero_grad, backward, clip_grad_norm_, optimizer.step.
                loss = None

                # TODO: Hard-clamp log_std at 0 (so σ ≤ 1) to prevent
                # σ runaway. See the numerical-stability note above.
                # Hint: self.actor.log_std.data.clamp_(max=...)
                pass

        return {
            "approx_kl": float(np.mean(approx_kls)) if approx_kls else 0.0,
            "mean_abs_adv": float(mean_abs_adv),
            "log_std": float(self.actor.log_std.detach().mean().item()),
        }

### Training loop

We collect rollouts CleanRL-style: `n_envs` parallel environment copies (built via SuperSuit's
`pettingzoo_env_to_vec_env_v1` + `concat_vec_envs_v1`), `rollout_steps` per copy, all flattened into
one batch for the PPO update. Each step advances `n_envs * n_agents` slots simultaneously, giving us
dense gradient signal per update.

We linearly anneal the learning rate from `hp.lr_*` toward $0$ over the full training run (CleanRL
convention), and we keep a snapshot of the actor/critic at the moment the 50-episode rolling-mean
return peaks — so the agent we return at the end is the best ever seen, not a late-stage drifted
copy.


In [ ]:
def train_mappo(aec_env: AECEnv, agent: MAPPO = None, num_updates: int = 500, seed: int = 0):
    """Train MAPPO with parallel envs, LR annealing, and best-model tracking.

    Wrapper stack:
      1. ``pettingzoo_env_to_vec_env_v1`` — treats each of the N agents in a
         single PettingZoo env as an independent slot in a gym VecEnv.
      2. ``concat_vec_envs_v1`` — replicates that VecEnv E times in-process;
         combined batch dim = E*N per step.

    At each update we (a) linearly anneal the LR on both optimizers from
    ``hp.lr_*`` toward 0 over ``num_updates`` iterations, (b) collect a
    step-based rollout, (c) compute per-slot GAE, (d) call ``agent.update``,
    (e) track the best actor/critic snapshot by 50-episode rolling mean.
    After training we restore that best snapshot into the agent so the
    returned agent is not a late-stage drifted copy.
    """
    # TODO: Convert the AECEnv to a vectorized gym env using SuperSuit.
    # 1. parallel_env = aec_to_parallel(aec_env)
    # 2. vec_env = ss.pettingzoo_env_to_vec_env_v1(parallel_env)
    # 3. envs = ss.concat_vec_envs_v1(vec_env, num_vec_envs=n_envs,
    #          num_cpus=0, base_class="gymnasium")
    parallel_env = None
    vec_env = None

    env_agent_ids = list(aec_env.possible_agents)
    n_agents = len(env_agent_ids)
    action_dim = int(np.prod(vec_env.action_space.shape))

    if agent is None:
        agent = MAPPO(env_agent_ids, action_dim)

    n_envs = agent.hp.n_envs
    rollout_steps = agent.hp.rollout_steps
    n_slots = n_envs * n_agents

    envs = ss.concat_vec_envs_v1(vec_env, num_vec_envs=n_envs, num_cpus=0, base_class="gymnasium")

    obs_shape = tuple(aec_env.observation_space(env_agent_ids[0]).shape)

    obs_buf = torch.zeros((rollout_steps, n_slots) + obs_shape, dtype=torch.uint8, device=DEVICE)
    # We store the *pre-squash* Gaussian sample, not the executed action. PPO needs to
    # re-evaluate log π on the same random variable during the update, so this is what
    # the rollout buffer must keep.
    actions_buf = torch.zeros(
        (rollout_steps, n_slots, action_dim), dtype=torch.float32, device=DEVICE
    )
    logprobs_buf = torch.zeros((rollout_steps, n_slots), dtype=torch.float32, device=DEVICE)
    rewards_buf = torch.zeros((rollout_steps, n_slots), dtype=torch.float32, device=DEVICE)
    dones_buf = torch.zeros((rollout_steps, n_slots), dtype=torch.float32, device=DEVICE)
    values_buf = torch.zeros((rollout_steps, n_slots), dtype=torch.float32, device=DEVICE)

    next_obs_np, _ = envs.reset(seed=seed)
    next_obs = torch.as_tensor(next_obs_np, device=DEVICE)
    next_done = torch.zeros(n_slots, dtype=torch.float32, device=DEVICE)

    slot_returns = np.zeros(n_slots, dtype=np.float64)
    returns = []

    # Best-model tracking: snapshot when 50-episode rolling mean sets a new max.
    best_rolling_mean = -float("inf")
    best_snapshot = None
    best_update = -1

    for update in range(1, num_updates + 1):
        # -------- LR annealing (linear, CleanRL-style) --------
        if agent.hp.anneal_lr:
            frac = 1.0 - (update - 1) / num_updates
            lr_actor_now = agent.hp.lr_actor * frac
            lr_critic_now = agent.hp.lr_critic * frac
            for pg in agent.opt_actor.param_groups:
                pg["lr"] = lr_actor_now
            for pg in agent.opt_critic.param_groups:
                pg["lr"] = lr_critic_now
        else:
            lr_actor_now = agent.hp.lr_actor

        # -------- Rollout --------
        for step in range(rollout_steps):
            obs_buf[step] = next_obs
            dones_buf[step] = next_done

            # TODO: Sample under no_grad. `_act_and_logprob` returns three things:
            # the *executed* action (already in (-1, 1) thanks to tanh-squash), the
            # *pre-squash* Gaussian sample (used by the buffer / actor update), and
            # the log-prob with the tanh correction applied.
            with torch.no_grad():
                action, pre_squash, logp = None, None, None
                values = None

            # Store the pre-squash sample (not the executed action) — see the
            # comment on `actions_buf` above and `MAPPO.update`.
            actions_buf[step] = pre_squash
            logprobs_buf[step] = logp
            values_buf[step] = values

            # Tanh keeps actions bounded — no extra `np.clip` needed.
            actions_env = action.detach().cpu().numpy()
            next_obs_np, rew_np, term_np, trunc_np, _ = envs.step(actions_env)

            rewards_buf[step] = torch.as_tensor(rew_np, dtype=torch.float32, device=DEVICE)
            done_np = np.logical_or(term_np, trunc_np).astype(np.float32)

            slot_returns += rew_np
            for c in range(n_envs):
                if done_np[c * n_agents]:
                    returns.append(
                        {
                            aid: float(slot_returns[c * n_agents + i])
                            for i, aid in enumerate(env_agent_ids)
                        }
                    )
                    slot_returns[c * n_agents : (c + 1) * n_agents] = 0.0

            next_obs = torch.as_tensor(next_obs_np, device=DEVICE)
            next_done = torch.as_tensor(done_np, dtype=torch.float32, device=DEVICE)

        # -------- GAE (per-slot) --------
        with torch.no_grad():
            next_value = agent.critic(next_obs)

        # TODO: Implement Generalized Advantage Estimation. For each timestep
        # t (iterated in REVERSE):
        #   nextnonterminal = 1 - dones at t+1 (or next_done at the boundary)
        #   nextvalues      = values at t+1   (or next_value at the boundary)
        #   delta = r_t + gamma * nextvalues * nextnonterminal - V_t
        #   adv_t = delta + gamma * gae_lambda * nextnonterminal * adv_{t+1}
        # Returns are then computed as (advantages + values_buf).
        advantages = torch.zeros_like(rewards_buf)
        # ... fill in advantages here ...
        returns_tensor = advantages + values_buf

        b_obs = obs_buf.reshape((rollout_steps * n_slots,) + obs_shape)
        b_actions = actions_buf.reshape(rollout_steps * n_slots, action_dim)
        b_logprobs = logprobs_buf.reshape(rollout_steps * n_slots)
        b_advantages = advantages.reshape(rollout_steps * n_slots)
        b_returns = returns_tensor.reshape(rollout_steps * n_slots)

        # TODO: Call agent.update(...) with the flattened buffers.
        diag = None

        # -------- Best-model checkpoint --------
        if len(returns) >= 50:
            rolling_mean = float(np.mean([np.mean(list(r.values())) for r in returns[-50:]]))
            if rolling_mean > best_rolling_mean:
                best_rolling_mean = rolling_mean
                best_snapshot = {
                    "actor": copy.deepcopy(agent.actor.state_dict()),
                    "critic": copy.deepcopy(agent.critic.state_dict()),
                }
                best_update = update

        if update % 5 == 0 and len(returns) >= 50:
            print(
                f"[Upd {update}/{num_updates} | eps {len(returns)}] "
                f"ret {rolling_mean:+7.3f} | "
                f"best {best_rolling_mean:+7.3f} @{best_update} | "
                f"log_std {diag['log_std']:+.2f} | "
                f"|adv| {diag['mean_abs_adv']:.2f} | "
                f"kl {diag['approx_kl']:+.4f} | "
                f"lr {lr_actor_now:.2e}".ljust(140)
            )

    # Restore best snapshot so the returned agent is the peak, not the last.
    if best_snapshot is not None:
        agent.actor.load_state_dict(best_snapshot["actor"])
        agent.critic.load_state_dict(best_snapshot["critic"])
        print(
            f"\nRestored best agent from update {best_update} "
            f"(50-episode rolling mean return: {best_rolling_mean:+.3f})"
        )

    return returns, agent

### Training MAPPO on Pistonball

For an apples-to-apples comparison with MADDPG, we train MAPPO on the **same `PISTONBALL_N5`
environment** (5 pistons, same physics, same reward structure). Same wrappers (grayscale, resize to
$32\times32$, frame-stack), same task — only the algorithm changes. This is the cleanest way to
attribute any performance difference to MAPPO vs MADDPG.

If you want to experiment with larger setups (`n_pistons=10` or `n_pistons=20`), be aware that
Pistonball-20 continuous is genuinely hard: the only published recipe that solves it cleanly uses a
frozen pretrained ResNet18 vision encoder and ~5M timesteps of training. Our small CNN-from-scratch
setup is not expected to scale to 20 pistons without similar tooling.


**Hyperparameter choice.** We use the
[RLlib Pistonball tutorial](https://pettingzoo.farama.org/tutorials/rllib/pistonball/) defaults for
continuous PPO: `lr=2e-5` with linear annealing, `clip_coef=0.4`, `gae_lambda=0.9`, `ent_coef=0.1`.
These are conservative compared to the standard CleanRL continuous-PPO config (`lr=3e-4`,
`clip_coef=0.2`) — Pistonball's noisy gradient signal benefits from smaller, more cautious updates
and a slightly stronger entropy bonus to keep exploration alive while the `log_std` clamp prevents σ
runaway.


In [ ]:
pv6_env_mappo: AECEnv = create_pistonball_env(**PISTONBALL_N5)
pv6_env_mappo = init_random(pv6_env_mappo)

agent_ids_mappo = pv6_env_mappo.possible_agents
action_dim_mappo = int(np.prod(pv6_env_mappo.action_space(agent_ids_mappo[0]).shape))

mappo_hp = MAPPOHyperParams(
    gamma=0.99,
    gae_lambda=0.9,
    clip_coef=0.4,
    ent_coef=0.1,
    lr_actor=2e-5,
    lr_critic=2e-5,
    anneal_lr=True,
    ppo_epochs=10,
    minibatch_size=2048,
    max_grad_norm=0.5,
    n_envs=10,
    rollout_steps=128,
)

agent_mappo = MAPPO(agent_ids_mappo, action_dim_mappo, hp=mappo_hp)

In [ ]:
# Train MAPPO with 10 parallel env copies (CleanRL-style step-based rollout).
#
# Recommended: GPU. With n_envs=10 and n_agents=5, each env.step() advances
# 50 agent slots. rollout_steps=128 collects 128 * 50 = 6_400 samples per
# PPO update across ~64 completed episodes (max_cycles=20 per episode).
# 650 updates total — the policy plateaus well before that, so we mainly
# coast through the linear LR-anneal schedule.

mappo_scores, agent_mappo = train_mappo(pv6_env_mappo, agent=agent_mappo, num_updates=650, seed=42)

In [ ]:
plot_agent_returns(mappo_scores)

In [ ]:
# Visualize the learned MAPPO policy

pettingzoo_simulation(
    pistonball_v6,
    agent=agent_mappo,
    wrappers=[pistonball_v6_wrappers],
    **PISTONBALL_N5,
)

### Results: MAPPO vs MADDPG on Pistonball-5

On the same `PISTONBALL_N5` environment, with the same wrappers and same task, we observed:

| Algorithm                                   | Best 50-episode rolling-mean return |
| ------------------------------------------- | ----------------------------------- |
| **MADDPG** (per-agent independent networks) | **+37.5**                           |
| **MAPPO** (parameter-shared single network) | **+87.9**                           |

MAPPO produces a **~2.3× higher return** on the identical task — a real, measurable improvement.

**What's the same.** Both deterministic policies converge to a degenerate "single piston up, rest
down" static formation. The specific piston that ends up "up" differs (rightmost in MAPPO,
second-from-rightmost in MADDPG), but neither algorithm produces the dynamic, rhythmic up/down
piston coordination that the term "Multi-Agent RL" intuitively suggests.

**Why the numerical gap then?** Likely a mix of:

1. **Position matters.** Rightmost-up is a better static formation than second-from-rightmost-up on
   Pistonball — the rightmost piston catches more of the random ball drops.
2. **MAPPO's stochastic exploration** (Gaussian noise around $\mu$ at $\sigma=1$) generates
   incidental ball-pushing motion during rollouts. MADDPG's noise-perturbed deterministic actor
   produces less of this kind of motion.
3. **Parameter sharing** lets MAPPO improve every piston's policy from every piston's experience.
   MADDPG's $N$ separate networks each learn from only their own slot, so each network sees roughly
   $1/N$ the effective gradient signal.

**Why doesn't either find dynamic coordination?** Pistonball's reward function only rewards leftward
ball motion, with no penalty for "piston staying up after the ball passes" and no per-agent reward
decomposition (the `local_ratio` parameter was removed in `pistonball_v6`). Under this reward
structure, a coordinated _static formation_ can be near-optimal — and static formations are much
easier to discover via random exploration than temporally coordinated dances.

**Pedagogical takeaway.** Even when MARL algorithms produce numerically different results on the
same task, the qualitative behavior of the learned _deterministic_ policies can be similar. MAPPO's
algorithmic advantages over MADDPG (parameter sharing for homogeneous agents, on-policy stability)
translate into measurable training-return improvements, but they don't fundamentally change what's
_achievable_ when the environment's reward landscape favors degenerate solutions. For a more
dramatic demonstration of dynamic coordination emerging in MARL, benchmarks like
[SMAC](https://github.com/oxwhirl/smac) or
[Hanabi](https://github.com/deepmind/hanabi-learning-environment) — where _no_ static policy can
solve the task — are better suited.


## Appendix: A Glimpse into the MARL Landscape

While we implemented MAPPO, the MARL field is vast. Here is a quick tour of other key approaches:

### 1. Independent Learning (IQL, IPPO)

- **Idea**: Each agent treats other agents as part of the environment and runs a standard
  single-agent algo (DQN, PPO).
- **Pros**: Extremely simple to implement; surprisingly effective baselines.
- **Cons**: Non-stationarity (learning agents change the environment dynamics for others), leading
  to instability.

### 2. Value Decomposition (VDN, QMIX)

Useful for **cooperative scenarios with discrete actions**.

- **Idea**: Decompose the global team reward into individual agent contributions.
- **VDN**: Simply sums up individual agent Q-values to estimate the total team Q-value:
  $Q_{tot} = \sum_i Q_i(o_i, a_i)$. This is easy to implement but can only represent additive value
  structures.
- **QMIX**: Improves on VDN by using a _mixing network_ (whose weights are conditioned on the global
  state) to combine individual Q-values non-linearly. The key constraint is **monotonicity**:
  $\partial Q_{tot} / \partial Q_i \ge 0$, which ensures that the greedy action for each agent under
  its own $Q_i$ is also greedy under $Q_{tot}$. This makes decentralized execution trivial while
  allowing richer coordination than simple summation.

### 3. Heterogeneous-Agent PPO (HAPPO)

Useful when agents are **heterogeneous** (different roles/capabilities).

- **Idea**: Unlike MAPPO's simultaneous updates, HAPPO updates agents **sequentially** (one by one).
  At each step, the advantage is decomposed so that agent $i$'s update accounts for the policy
  changes already made by agents $1 \ldots i-1$.
- **Why**: This sequential decomposition provides a **monotonic improvement guarantee** — the joint
  policy is guaranteed to not get worse after each agent's update. Simultaneous updates (as in
  MAPPO) lack this guarantee and can lead to "coordination failure" where individually reasonable
  updates combine to hurt the team.

### Where to go from here?

- **RNN-MAPPO**: Add GRU/LSTM to the actor for Partial Observability.
- **Self-Play**: If you want to train competitive agents (like the Adversary in MPE).
- **QMIX**: If you have a discrete action space and want to see state-of-the-art value
  decomposition.
